# 01 — Download + parse NHANES `P_LUX` and `P_DEMO`

Pull the two SAS XPT files for the 2017 – March 2020 pre-pandemic combined release (already cached at `data/raw/nhanes/2017_2020_prepandemic/`), keep just the variables this analysis needs, and save a tidy parquet for the next notebook.

Variables kept:

- `SEQN` — respondent ID (join key)
- `RIAGENDR` — sex (1 = Male, 2 = Female)
- `RIDAGEYR` — age in years (integer; topcode at 80 in continuous NHANES)
- `WTMECPRP` — MEC pre-pandemic examination weight
- `SDMVPSU`, `SDMVSTRA` — masked variance pseudo-PSU and stratum
- `LUXSMED` — median liver stiffness (kPa) — primary outcome
- `LUXSIQR` — IQR of valid stiffness measurements
- `LUAXSTAT` — examination status flag (`1` = complete; we keep only complete exams)

In [1]:
import os, urllib.request
import numpy as np
import pandas as pd
from pathlib import Path

DATA_DIR = Path(os.path.abspath(os.path.join('..', 'data')))
RAW_DIR = DATA_DIR / 'raw' / 'nhanes' / '2017_2020_prepandemic'
DERIVED_DIR = DATA_DIR / 'derived'
RAW_DIR.mkdir(parents=True, exist_ok=True)
DERIVED_DIR.mkdir(parents=True, exist_ok=True)

FILES = {
    'P_DEMO.xpt': 'https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_DEMO.xpt',
    'P_LUX.xpt':  'https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_LUX.xpt',
}

In [2]:
for fname, url in FILES.items():
    out = RAW_DIR / fname
    if out.exists():
        print(f'cached: {out} ({out.stat().st_size:,} bytes)')
        continue
    print(f'Downloading {url} → {out}...')
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=120) as r:
        out.write_bytes(r.read())
    print(f'  saved {out.stat().st_size:,} bytes')

cached: /home/abie/csu_mace_rct_sim/ai_assisted_us_health_data_analysis/data/raw/nhanes/2017_2020_prepandemic/P_DEMO.xpt (3,614,720 bytes)
cached: /home/abie/csu_mace_rct_sim/ai_assisted_us_health_data_analysis/data/raw/nhanes/2017_2020_prepandemic/P_LUX.xpt (1,105,920 bytes)


## Parse and merge

`pd.read_sas` reads the XPT directly. We keep only the columns of interest and merge on `SEQN`.

In [3]:
DEMO_COLS = ['SEQN', 'RIAGENDR', 'RIDAGEYR', 'WTMECPRP', 'SDMVPSU', 'SDMVSTRA']
LUX_COLS  = ['SEQN', 'LUXSMED', 'LUXSIQR', 'LUAXSTAT']

demo = pd.read_sas(RAW_DIR / 'P_DEMO.xpt')[DEMO_COLS]
lux  = pd.read_sas(RAW_DIR / 'P_LUX.xpt')[LUX_COLS]

df = demo.merge(lux, on='SEQN', how='left')
print(f'parsed: {len(df):,} respondents (DEMO), {len(lux):,} elastography records')
print(f'  with LUXSMED: {df["LUXSMED"].notna().sum():,}')
df.head()

parsed: 15,560 respondents (DEMO), 10,409 elastography records
  with LUXSMED: 9,700


,SEQN,RIAGENDR,RIDAGEYR,WTMECPRP,SDMVPSU,SDMVSTRA,LUXSMED,LUXSIQR,LUAXSTAT
0,109263.0,1.0,2.0,8.951816e+03,3.0,156.0,NaN,NaN,NaN
1,109264.0,2.0,13.0,1.227116e+04,1.0,155.0,NaN,NaN,3.0
2,109265.0,1.0,2.0,1.665876e+04,1.0,157.0,NaN,NaN,NaN
3,109266.0,2.0,29.0,8.154968e+03,2.0,168.0,6.4,1.0,1.0
4,109267.0,2.0,21.0,5.397605e-79,1.0,156.0,NaN,NaN,NaN


In [4]:
# Sex / status labels and integer coercions.
df['SEQN'] = df['SEQN'].astype(int).astype(str)
df['sex'] = df['RIAGENDR'].map({1.0: 'Male', 2.0: 'Female'})
df['age_years'] = df['RIDAGEYR'].astype(float)
df['exam_complete'] = df['LUAXSTAT'] == 1.0
df['LSM_KPA'] = df['LUXSMED']
df['LSM_IQR'] = df['LUXSIQR']

for col in ['SDMVPSU', 'SDMVSTRA']:
    df[col] = df[col].astype('Int64')
df['WTMECPRP'] = df['WTMECPRP'].astype(float)
df.head()

,SEQN,RIAGENDR,RIDAGEYR,WTMECPRP,SDMVPSU,SDMVSTRA,LUXSMED,LUXSIQR,LUAXSTAT,sex,age_years,exam_complete,LSM_KPA,LSM_IQR
0,109263,1.0,2.0,8.951816e+03,3,156,NaN,NaN,NaN,Male,2.0,False,NaN,NaN
1,109264,2.0,13.0,1.227116e+04,1,155,NaN,NaN,3.0,Female,13.0,False,NaN,NaN
2,109265,1.0,2.0,1.665876e+04,1,157,NaN,NaN,NaN,Male,2.0,False,NaN,NaN
3,109266,2.0,29.0,8.154968e+03,2,168,6.4,1.0,1.0,Female,29.0,True,6.4,1.0
4,109267,2.0,21.0,5.397605e-79,1,156,NaN,NaN,NaN,Female,21.0,False,NaN,NaN


In [5]:
print('Total P_DEMO rows           :', f'{len(df):,}')
print('Has LSM measurement         :', f'{df["LSM_KPA"].notna().sum():,}')
print('  — exam_complete           :', f'{(df["exam_complete"] & df["LSM_KPA"].notna()).sum():,}')
print('  — adults 20+              :', f'{(df["exam_complete"] & df["LSM_KPA"].notna() & (df["age_years"] >= 20)).sum():,}')
print('  — trial-band 65-80        :', f'{(df["exam_complete"] & df["LSM_KPA"].notna() & (df["age_years"] >= 65) & (df["age_years"] <= 80)).sum():,}')
print()
print('LSM (kPa) summary, MEC-examined adults 20+:')
ad = df[df['exam_complete'] & df['LSM_KPA'].notna() & (df['age_years'] >= 20)]
print(ad[['LSM_KPA', 'LSM_IQR']].describe().round(2))

Total P_DEMO rows           : 15,560
Has LSM measurement         : 9,700
  — exam_complete           : 9,023
  — adults 20+              : 7,396
  — trial-band 65-80        : 1,787

LSM (kPa) summary, MEC-examined adults 20+:
       LSM_KPA  LSM_IQR
count  7396.00  7396.00
mean      5.83     0.83
std       4.50     0.79
min       1.60     0.00
25%       4.10     0.40
50%       5.00     0.70
75%       6.20     1.00
max      75.00    17.30


In [6]:
out = DERIVED_DIR / 'nhanes_p_lux.parquet'
keep = ['SEQN', 'sex', 'age_years', 'WTMECPRP', 'SDMVPSU', 'SDMVSTRA',
        'LSM_KPA', 'LSM_IQR', 'exam_complete']
df[keep].to_parquet(out, index=False)
print(f'wrote {out} ({out.stat().st_size:,} bytes)')
df[keep].dtypes

wrote /home/abie/csu_mace_rct_sim/ai_assisted_us_health_data_analysis/data/derived/nhanes_p_lux.parquet (285,853 bytes)


SEQN              object
sex               object
age_years        float64
WTMECPRP         float64
SDMVPSU            Int64
SDMVSTRA           Int64
LSM_KPA          float64
LSM_IQR          float64
exam_complete       bool
dtype: object